In [ ]:
import os
import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import inv_boxcox
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics.pairwise import haversine_distances
import matplotlib.pyplot as plt
import joblib

# ---------------------------
# 1️⃣ Carga y limpieza
# ---------------------------
def load_and_clean_data(path_csv):
    df = pd.read_csv(path_csv)
    cols_to_drop = ["id", "name", "host_id", "host_name", "last_review", "license"]
    df = df.drop(cols_to_drop, axis=1, errors='ignore')
    df = df.drop_duplicates()
    df = df.dropna()
    print(f"Dataset limpio: {df.shape}")
    return df

# ---------------------------
# 2️⃣ Feature Engineering
# ---------------------------
def feature_engineering(df):
    df['availability_ratio'] = df['availability_365'] / 365
    df['professional_host'] = (df['calculated_host_listings_count'] > 1).astype(int)
    return df

# ---------------------------
# 3️⃣ Train/Test split
# ---------------------------
def split_data(df, target='price', test_size=0.2, random_state=42):
    X = df.drop(target, axis=1)
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    print(f"Tamaño entrenamiento: {X_train.shape}, prueba: {X_test.shape}")
    return X_train, X_test, y_train, y_test

# ---------------------------
# 4️⃣ Preprocesamiento
# ---------------------------
def preprocess(X_train, X_test):
    # --- One-hot encoding ---
    col_dummies = ['neighbourhood_group', 'neighbourhood']
    X_train = pd.get_dummies(X_train, columns=col_dummies, drop_first=True)
    X_test  = pd.get_dummies(X_test,  columns=col_dummies, drop_first=True)
    X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)
    
    # --- Room type ordinal ---
    mapeo_room = {"Hotel room": 2, "Entire home/apt": 3, "Private room": 1, "Shared room": 0}
    X_train["room_type"] = X_train["room_type"].map(mapeo_room)
    X_test["room_type"] = X_test["room_type"].map(mapeo_room)
    
    # --- Distance to center ---
    madrid_center = np.radians([[40.4168, -3.7038]])
    coords_train = np.radians(X_train[["latitude","longitude"]])
    coords_test  = np.radians(X_test[["latitude","longitude"]])
    X_train["distance_to_center_km"] = haversine_distances(coords_train, madrid_center)*6371
    X_test["distance_to_center_km"]   = haversine_distances(coords_test, madrid_center)*6371
    X_train = X_train.drop(columns=["latitude","longitude"])
    X_test  = X_test.drop(columns=["latitude","longitude"])
    
    # --- Log transformation ---
    columnas_log = ['reviews_per_month', 'calculated_host_listings_count',
                    'distance_to_center_km', 'number_of_reviews', 'minimum_nights']
    X_train[columnas_log] = X_train[columnas_log].apply(np.log1p)
    X_test[columnas_log]  = X_test[columnas_log].apply(np.log1p)
    
    # --- Escalado (solo variables de alta cardinalidad) ---
    threshold = 10
    high_cardinality = [col for col in X_train.columns if X_train[col].nunique() > threshold]
    scaler = StandardScaler()
    X_train[high_cardinality] = scaler.fit_transform(X_train[high_cardinality])
    X_test[high_cardinality]  = scaler.transform(X_test[high_cardinality])
    
    return X_train, X_test

# ---------------------------
# 5️⃣ Box-Cox objetivo
# ---------------------------
def boxcox_transform(y_train, y_test):
    y_train_bc, lambda_bc = stats.boxcox(y_train)
    y_test_bc = stats.boxcox(y_test, lmbda=lambda_bc)
    return y_train_bc, y_test_bc, lambda_bc

# ---------------------------
# 6️⃣ Carga modelos y predicción ensemble
# ---------------------------
def predict_ensemble(X_test, models_paths):
    y_preds_real = []
    y_preds_bc   = []
    for m_path in models_paths:
        m = joblib.load(m_path)
        model = m["model"]
        features = m["features"]
        lambda_bc = m["lambda_bc"]
        y_pred_bc = model.predict(X_test[features])
        y_pred_real = inv_boxcox(y_pred_bc, lambda_bc)
        y_preds_real.append(y_pred_real)
        y_preds_bc.append(y_pred_bc)
    y_pred_real_ensemble = np.mean(y_preds_real, axis=0)
    y_pred_bc_ensemble   = np.mean(y_preds_bc, axis=0)
    return y_pred_real_ensemble, y_pred_bc_ensemble

# ---------------------------
# 7️⃣ Evaluación
# ---------------------------
def evaluate(y_test, y_test_bc, y_pred_real, y_pred_bc):
    mae  = mean_absolute_error(y_test, y_pred_real)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred_real))
    r2   = r2_score(y_test_bc, y_pred_bc)
    print(f"MAE:  {mae:.2f} €")
    print(f"RMSE: {rmse:.2f} €")
    print(f"R²:   {r2:.4f}")
    return mae, rmse, r2

# ---------------------------
# 8️⃣ Visualización predicciones
# ---------------------------
def plot_preds(y_test, y_pred_real, title="Predicciones vs reales"):
    plt.figure(figsize=(6,6))
    plt.scatter(y_test, y_pred_real, alpha=0.5, color='dodgerblue')
    plt.plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()],
             'r--', lw=2)
    plt.xlabel("Precio real (€)")
    plt.ylabel("Predicción (€)")
    plt.title(title)
    plt.grid(True)
    plt.show()

# ---------------------------
# 9️⃣ Pipeline completo
# ---------------------------
def pipeline(df_path, model_paths):
    df = load_and_clean_data(df_path)
    df = feature_engineering(df)
    X_train, X_test, y_train, y_test = split_data(df)
    X_train, X_test = preprocess(X_train, X_test)
    y_train_bc, y_test_bc, lambda_bc = boxcox_transform(y_train, y_test)
    y_pred_real, y_pred_bc = predict_ensemble(X_test, model_paths)
    evaluate(y_test, y_test_bc, y_pred_real, y_pred_bc)
    plot_preds(y_test, y_pred_real)

# ---------------------------
# 10️⃣ Uso
# ---------------------------
models_paths = [
    r"modelo_airbnb_lr.pkl",
    r"modelo_airbnb_rf.pkl",
    r"modelo_airbnb_xg.pkl",
    r"modelo_airbnb_lgbm.pkl"
]

pipeline("df_precios.csv", models_paths)